In [1]:
import pandas as pd
import json

In [2]:
first_file_num, last_file_num = 0, 10
df = pd.DataFrame()
for num in range(first_file_num, last_file_num + 1):
    with open(f"extracted_images/whole_{num}.json") as f:
        contents = f.read()
        file_json = json.loads(contents)
        file_df = pd.DataFrame(file_json)
    df = pd.concat([df, file_df], ignore_index=True)


In [3]:
df['book_id'] = df.id.apply(lambda x: x[:10])
df['page_num'] = df.id.apply(lambda x: x[10:15].strip('0'))

In [5]:
df.sample(10)

,id,labels,page_labels,scores,boxes,book_id,page_num
161223,014030010203390.TIF,12,"[12, 12, 1, 12]",0.990184,"[44.0911979675293, 1806.3883056640625, 2203.89...",0140300102,339
226887,062810030004230.TIF,3,"[3, 3, 2, 7, 3, 1, 2]",0.831898,"[899.1222534179688, 294.6745300292969, 944.950...",0628100300,423
70426,016210050102070.TIF,12,"[7, 12, 7, 12]",0.497385,"[18.696311950683594, 473.69940185546875, 1927....",0162100501,207
3492,023680060003130.TIF,1,"[1, 1, 3]",0.993234,"[53.0727424621582, 226.6192169189453, 1048.726...",0236800600,313
198621,013030020000270.TIF,1,"[1, 1, 1, 1]",0.971316,"[0.0, 1267.5662841796875, 1254.9921875, 1287.6...",0130300200,27
34525,039050010803030.TIF,1,[1],0.992549,"[112.27224731445312, 128.22576904296875, 1182....",0390500108,303
123705,093580030000100.TIF,7,"[15, 7, 1]",0.999338,"[69.07551574707031, 275.06011962890625, 983.79...",0935800300,1
144718,056490010103080.TIF,3,"[2, 2, 3, 3, 3, 1]",0.370001,"[1118.9791259765625, 1661.4669189453125, 1144....",0564900101,308
28999,017590120000010.TIF,1,[1],0.993085,"[327.0096435546875, 2009.3326416015625, 603.01...",0175901200,1
39586,069030010305080.TIF,2,"[2, 2, 3, 2, 3, 3, 3, 3, 6, 6, 6, 6, 1, 1, 1, ...",0.932037,"[1885.15625, 2584.72314453125, 1904.0490722656...",0690300103,508


### Label Meanings

1: ‘FI-HORIZONTAL RULE’, 

2: ‘FI-MISC. FUNCTIONAL ELEMENT’, 

3: ‘FI-VERTICAL RULE’, 

4: ‘O-LIBRARY STAMP’, 

5: ‘O-MISCELLANEOUS OTHER’, 

6: ‘PO-BORDER’, 

7: ‘PO-HEADPIECE’, 

8: ‘PO-OTHER PRINTER’S ORNAMENT’, 

9: ‘PO-PRINTERS DEVICE’, 

10: ‘PO-TAILPIECE’, 

11: ‘ILLU-FRONTISPIECE’, 

12: ‘ILLU-WOODCUT OR ENGRAVING’, 

13: ‘ILLU-OTHER’, 

14: ‘INIT-DECORATIVE INITIAL’, 

15: ‘INIT-FACTOTUM’

In [147]:
illustr = df[df.labels.isin([5, 8, 11, 12, 13])]
illustr_counts = illustr.book_id.value_counts().to_dict()

# Subsets

In [16]:
links = pd.read_csv("estc-data-unified/estc-eebo-ecco-links/idpairs_ecco_eebo_estc.csv")
works = pd.read_csv("estc-data-unified/estc-work-id/estc_work_id.tsv", sep='\t')
books = pd.read_csv("estc_student_edition.csv")
books["full_title"] = books.apply(lambda row: (str(row.title) + " " + str(row.remaining_title)).lower(), axis=1)

/var/folders/qn/_b4cqwl14qv5mxs65p_y4zhh0000gn/T/ipykernel_19264/4184046220.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  links = pd.read_csv("estc-data-unified/estc-eebo-ecco-links/idpairs_ecco_eebo_estc.csv")


In [ ]:
manual_heraldry_book_ids = ['R26135', 'R181126', 'R12114', 'N22559', 'N17450', 'T94010', 
                  'T166153', 'R32568', 'T100037', 'T113231', 'T117647', 'T134649', 
                  'T135745', 'T137070', 'T140947', 'T153613', 'T155315', 'T166154'
                  'T200312', 'T55579', 'T77924', 'T83445', 'T94006']

In [170]:
def get_heraldry_books_info(book_student_ids):
    get_estc_id = lambda student_id: links[links.estc_id_student_edition == student_id].estc_id.values[0]
    def get_document_id(estc_id, student_id): 
        link_data = links[links.estc_id == estc_id].document_id.values
        if not link_data.any():
            link_data = links[links.estc_id_student_edition == student_id].document_id.values
        if not link_data.any():
            return None
        return link_data[0]
    get_student_id = lambda estc_id: links[links.estc_id == estc_id].estc_id_student_edition.values[0] if estc_id in links.estc_id.values else None
    get_publ_year = lambda student_id: books[books.id == student_id].publication_year.values[0] if student_id in books.id.values else None
    get_pagecount = lambda student_id: books[books.id == student_id].pagecount.values[0] if student_id in books.id.values else None
    df = pd.DataFrame()
    for sid in book_student_ids:
        if sid not in links.estc_id_student_edition.values:
            continue
        estc_id = get_estc_id(sid)
        work_id = works[works.estc_id == estc_id]['work_id'].values[0] if estc_id in works.estc_id.values else None
        editions = works[works.work_id == work_id]['estc_id'].values if work_id else [estc_id]
        edition_doc_ids = [get_document_id(eid, sid) for eid in editions]
        edition_student_ids = [get_student_id(eid) for eid in editions]
        publ_years = [get_publ_year(sid) for sid in edition_student_ids]
        pagecounts = [get_pagecount(sid) for sid in edition_student_ids]
        illustr_num = [illustr_counts[str(did)] if str(did) in illustr_counts.keys() 
                       else illustr_counts[int(did)] if str(did).isnumeric() and int(did) in illustr_counts.keys() 
                       else None for did in edition_doc_ids]
        illustr_per_page = []
        for i, n in enumerate(illustr_num):
            try:
                ill_page = n / pagecounts[i]
            except:
                ill_page = None
            illustr_per_page.append(ill_page)
        work_df = pd.DataFrame({"work_id": work_id, "edition_estc_id": estc_id, "edition_document_id": edition_doc_ids, 
                                "edition_estc_id_student_edition": edition_student_ids, "publication_year": publ_years, 
                                "num_of_illustrations": illustr_num, "num_of_pages": pagecounts, "avg_illustrations_per_page": illustr_per_page})
        df = pd.concat([df, work_df], ignore_index=True)
    return df
    

In [138]:
keyword = 'Heraldry'
books = books[~books.subject_topic.isna()]
heraldry_books = books[books['subject_topic'].str.contains(keyword)]
heraldry_books_ids = heraldry_books.id.unique()

In [171]:
heraldry_books_info = get_heraldry_books_info(heraldry_books_ids)
heraldry_books_info

,work_id,edition_estc_id,edition_document_id,edition_estc_id_student_edition,publication_year,num_of_illustrations,num_of_pages,avg_illustrations_per_page
0,None,N001118,435700300,N1118,1793.0,None,118.0,NaN
1,None,N012198,109900100,N12198,1767.0,None,348.0,NaN
2,None,N017450,1274000200,N17450,1791.0,727,128.0,5.679688
3,None,N017665,1153000200,N17665,1795.0,38,325.0,0.116923
4,None,N020017,249200101,N20017,1789.0,None,2755.0,None
...,...,...,...,...,...,...,...,...
426,None,T094007,189600800,T94007,1771.0,None,332.0,None
427,None,T094008,535800800,T94008,1777.0,None,372.0,None
428,None,T094009,183100400,T94009,1787.0,None,388.0,None
429,None,T094010,515500400,T94010,1795.0,None,388.0,None


In [175]:
heraldry_books_info.to_csv("heraldry_books_info.csv")

In [6]:
import pandas as pd
heraldry_books_info = pd.read_csv("heraldry_books_info.csv", index_col=0)
heraldry_books_info["publication_year"].value_counts().sort_index()

1496.0    2
1566.0    2
1572.0    1
1595.0    2
1596.0    3
         ..
1793.0    4
1794.0    4
1795.0    4
1797.0    2
1800.0    1
Name: publication_year, Length: 85, dtype: int64

In [11]:
heraldry_books_w_imgs = heraldry_books_info[~heraldry_books_info["num_of_illustrations"].isna()][~heraldry_books_info["num_of_pages"].isna()]
heraldry_books_w_imgs.drop_duplicates(["edition_document_id", "publication_year", "num_of_illustrations", "num_of_pages"], inplace=True)

/var/folders/qn/_b4cqwl14qv5mxs65p_y4zhh0000gn/T/ipykernel_19264/500191446.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  heraldry_books_w_imgs = heraldry_books_info[~heraldry_books_info["num_of_illustrations"].isna()][~heraldry_books_info["num_of_pages"].isna()]


In [14]:
heraldry_books_w_imgs

,work_id,edition_estc_id,edition_document_id,edition_estc_id_student_edition,publication_year,num_of_illustrations,num_of_pages,avg_illustrations_per_page
2,NaN,N017450,1274000200,N17450,1791.0,727.0,128.0,5.679688
3,NaN,N017665,1153000200,N17665,1795.0,38.0,325.0,0.116923
6,NaN,N028522,1233500500,N28522,1739.0,108.0,18.0,6.000000
7,NaN,N032743,1097700100,N32743,1751.0,199.0,606.0,0.328383
12,X-heraldry in miniature,N33130,1705302200,N33130,1797.0,796.0,40.0,19.900000
13,X-heraldry in miniature,N33130,1014800300,T185743,1777.0,1092.0,44.0,24.818182
16,NaN,N003839,1262801200,N3839,1719.0,390.0,398.0,0.979899
160,15112-ducatus leodiensis or topography of anci...,T100187,1121900600,T142791,1715.0,159.0,671.0,0.236960
165,166933-peerage of great britain and ireland in...,T105741,1002600700,T105741,1793.0,74.0,202.0,0.366337
177,X-the heraldry of nature,T113785,1238001400,T182327,1785.0,96.0,96.0,1.000000


In [ ]:
import requests
import math
import os


def extract_imgs_from_book(row):
    doc_id = row['edition_document_id']
    page_num = int(row['num_of_pages']) if not math.isnan(row['num_of_pages']) else 0
    for p in range(1, page_num + 1):
        zero_separator = '000' if p < 10 else '00' if p < 100 else '0'
        img_url = f"https://oma-sivu.2.rahtiapp.fi/image/{doc_id}{zero_separator}{p}0"
        cur_dir = os.getcwd()
        book_dir =  os.path.join(cur_dir, f"images/{doc_id}")
        img_name =  os.path.join(book_dir, f"{doc_id}_{p}.png")
        try:
            img_data = requests.get(img_url).content
            if not os.path.exists(book_dir):
                os.makedirs(book_dir)
            with open(img_name, 'xb') as handler:
                handler.write(img_data)
            print(img_name)
        except Exception as e:
            print(e)
            continue

heraldry_books_w_imgs.apply(extract_imgs_from_book, axis=1)

In [ ]:
image_info_subset = df[df['book_id'].isin(heraldry_books_w_imgs.edition_document_id.values)]

# Move all images in a book to a separate folder
def separate_images(row):
    book = row.book_id
    page = row.page_num
    cur_dir = os.getcwd()
    book_dir = os.path.join(cur_dir, f"images/{book}")
    img_name = os.path.join(book_dir, f"{book}_{page}.png")
    subimg_dir = os.path.join(book_dir, f"selected_imgs")
    if os.path.exists(img_name):
        if not os.path.exists(subimg_dir):
            os.makedirs(subimg_dir)
        os.rename(img_name, os.path.join(subimg_dir, f"{book}_{page}.png"))

image_info_subset.apply(separate_images, axis=1)

385324      None
385325      None
385326      None
385327      None
385328      None
            ... 
15698626    None
15698627    None
15698628    None
15698629    None
15698630    None
Length: 12731, dtype: object

## Relevant books and images

In [ ]:
catalogues_ids = ['1274000200', '1014800300', '1097700100', '1121900300', 
                  '1121900600', '1153000200', '1221100200', '1233500500', 
                  '1238001400', '1262801200', '1266500200', '1712301301']

In [12]:
heraldry_books_w_imgs[heraldry_books_w_imgs["edition_document_id"].isin(catalogues_ids)]

,work_id,edition_estc_id,edition_document_id,edition_estc_id_student_edition,publication_year,num_of_illustrations,num_of_pages,avg_illustrations_per_page
2,NaN,N017450,1274000200,N17450,1791.0,727.0,128.0,5.679688
3,NaN,N017665,1153000200,N17665,1795.0,38.0,325.0,0.116923
6,NaN,N028522,1233500500,N28522,1739.0,108.0,18.0,6.000000
7,NaN,N032743,1097700100,N32743,1751.0,199.0,606.0,0.328383
13,X-heraldry in miniature,N33130,1014800300,T185743,1777.0,1092.0,44.0,24.818182
16,NaN,N003839,1262801200,N3839,1719.0,390.0,398.0,0.979899
160,15112-ducatus leodiensis or topography of anci...,T100187,1121900600,T142791,1715.0,159.0,671.0,0.236960
177,X-the heraldry of nature,T113785,1238001400,T182327,1785.0,96.0,96.0,1.000000
219,50365-grammar of heraldry or gentlemans vade m...,T117647,1221100200,T166154,1724.0,1571.0,250.0,6.284000
298,X-a new dictionary of heraldry,T134649,1266500200,T200312,1725.0,23.0,364.0,0.063187


In [14]:
import os

image_info_subset = df[df['book_id'].isin(catalogues_ids)]
for bid in catalogues_ids:
    cur_dir = os.getcwd()
    book_dir = os.path.join(cur_dir, f"images/{bid}/selected_imgs")
    dataset_dir = os.path.join(cur_dir, f"img_dataset")
    all_files = os.listdir(book_dir)
    for f in all_files:
        file_path = os.path.join(book_dir, f)
        os.popen(f'cp {file_path} {dataset_dir}')

In [15]:
heraldry_books_w_imgs[heraldry_books_w_imgs["edition_document_id"].isin(catalogues_ids)].to_csv("selected_catalogues.csv")

In [26]:
books.columns

Index(['id', 'title', 'remaining_title', 'document_type', 'subject_topic',
       'estc_language_vec', 'publication_year', 'publication_decade',
       'gatherings', 'width', 'height', 'area', 'pagecount', 'volcount',
       'paper', 'publication_place_752', 'country_752', 'longitude',
       'latitude', 'finalWorkField', 'WF_startDate', 'WF_endDate', 'actor_id',
       'is_organization', 'name_unified', 'actor_roles_all', 'full_title'],
      dtype='object')

In [ ]:
books[books.id == 'N33130']

62312    40.0
62313    40.0
62314    40.0
62315    40.0
62316    40.0
62317    40.0
62318    40.0
62319    40.0
62320    40.0
62321    40.0
62322    40.0
62323    40.0
62324    40.0
62325    40.0
62326    40.0
62327    40.0
62328    40.0
62329    40.0
Name: pagecount, dtype: float64

In [61]:
links[links.document_id == "1359700100"]

,estc_id,document_id,document_id_octavo,collection,estc_id_student_edition
162301,N63315,1359700100,1359700100,ecco,N63315


In [20]:
works.head()

,work_id,estc_id
0,17273-monthly review or literary journal givin...,P1965
1,58177-julia or new eloisa series of original l...,N52582
2,8345-gentlemans magazine,P1955
3,45364-oeuvres dhorace en latin et en francois ...,T13700
4,11117-british merchant or commerce preservd in...,P2147


In [50]:
kearsley = works[works.work_id == works[works.estc_id == 'T167876'].work_id.values[0]].estc_id.values
links[links.estc_id.isin(kearsley)]

,estc_id,document_id,document_id_octavo,collection,estc_id_student_edition
19036,T121551,117500500,117500500,ecco,T121551
30782,T121269,186600401,186600401,ecco,T121269
30783,T121269,186600402,186600402,ecco,T121269
100112,T121279,764700201,764700201,ecco,T121279
100113,T121279,764700202,764700202,ecco,T121279
162301,N63315,1359700100,1359700100,ecco,N63315
189079,T167876,1587300601,1587300601,ecco,T167876
189080,T167876,1587300602,1587300602,ecco,T167876


In [52]:
books[books.id.isin(links[links.estc_id.isin(kearsley)].estc_id.values)].drop_duplicates(["id", "publication_year", "pagecount"])[["id", "title", "publication_year", "pagecount"]]

,id,title,publication_year,pagecount
153478,N63315,"Kearsley's complete peerage, of England, Scotl...",1791.0,483.0
576471,T121269,Kearsley's complete peerage,1799.0,632.0
576497,T121279,Kearsley's complete peerage,1796.0,584.0
577334,T121551,Kearsley's complete peerage,1794.0,531.0
685061,T167876,"Kearsley's complete peerage, of England, Scotl...",1800.0,548.0


In [ ]:
import requests
import os

relevant_fields = ['title', 'remaining_title', 'publication_year', 'pagecount']
#selected_books_df = pd.DataFrame()


def extract_imgs_from_additional_book(doc_id):
    estc_id = links[links.document_id == doc_id]
    if estc_id.empty:
        estc_id = links[links.document_id == int(doc_id)]
    estc_id = estc_id.estc_id_student_edition.values[0]
    work_id = works[works.estc_id == estc_id].work_id.values[0]
    book_info = books[books.id == estc_id].drop_duplicates(["id", "publication_year", "pagecount"])[relevant_fields]
    page_num = int(book_info["pagecount"].values[0])
    pub_year = int(book_info["publication_year"].values[0])
    title = book_info["title"].values[0]
    remaining_title = book_info["remaining_title"].values[0]
    
    for p in range(1, page_num + 1):
        zero_separator = '000' if p < 10 else '00' if p < 100 else '0'
        img_url = f"https://oma-sivu.2.rahtiapp.fi/image/{doc_id}{zero_separator}{p}0"
        cur_dir = os.getcwd()
        book_dir =  os.path.join(cur_dir, f"images/{doc_id}")
        img_name =  os.path.join(book_dir, f"{doc_id}_{p}.png")
        try:
            img_data = requests.get(img_url).content
            if not os.path.exists(book_dir):
                os.makedirs(book_dir)
            with open(img_name, 'wb') as handler:
                handler.write(img_data)
            print(img_name)
        except Exception as e:
            print(e)
            continue
    
    doc_df = pd.DataFrame({"work_id": [work_id], "edition_estc_id": [estc_id], "edition_document_id": [doc_id], 
                               "title": [title], "remaining_title": [remaining_title],
                               "publication_year": [pub_year], "num_of_pages": [page_num]})
    global selected_books_df
    selected_books_df = pd.concat([selected_books_df, doc_df], ignore_index=True)

In [ ]:
catalogues_with_supporters_ids = ['1014800300', '1097700100', '1121900300', '1221100200', '1238001400', '1274000200', '1712301301']
for doc in catalogues_with_supporters_ids:
    extract_imgs_from_additional_book(doc)

In [78]:
selected_books_df

,work_id,edition_estc_id,edition_document_id,title,remaining_title,publication_year,num_of_pages
0,X-heraldry in miniature,T185743,1014800300,Heraldry in miniature,"containing the arms, crests, supporters, and m...",1777,44
1,1576-british compendium,N32743,1097700100,The British compendium,"or, rudiments of honour. Vol. I. In two parts....",1751,606
2,X-heraldry in miniature,T83865,1121900300,Heraldry in miniature,"containing all the arms, crests, supporters, a...",1788,176
3,50365-grammar of heraldry or gentlemans vade m...,T166154,1221100200,The grammar of heraldry,"or, gentleman's vade mecum, In which are laid ...",1724,250
4,X-the heraldry of nature,T182327,1238001400,The heraldry of nature,"or, instructions for the King at Arms: compris...",1785,96
5,X-heraldry in miniature,N17450,1274000200,Heraldry in miniature,"containing all the arms, crests, supporters an...",1791,128
6,1576-british compendium,T77908,1712301301,"The British compendium: or, Rudiments of honour","Containing the titles, descents, marriages, is...",1723,560
7,3098-kearsleys complete peerage of england sco...,T121551,117500500,Kearsley's complete peerage,"of England, Scotland and Ireland; together wit...",1794,531
8,3098-kearsleys complete peerage of england sco...,T121269,186600401,Kearsley's complete peerage,"of England, Scotland and Ireland; together wit...",1799,632
9,3098-kearsleys complete peerage of england sco...,T121279,764700201,Kearsley's complete peerage,"of England, Scotland and Ireland; together wit...",1796,584


In [ ]:
selected_books_df.to_csv("selected_catalogues.csv")
# NB: All published in London
# 568 image

In [84]:
additional_docs = ['117500500', '186600401', '764700201'] #'1359700100', '1587300601', '1587300602'

for doc in additional_docs:
    extract_imgs_from_additional_book(doc)

/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_1.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_2.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_3.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_4.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_5.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_6.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_7.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_8.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_9.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_10.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_11.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_12.png
/Users/nesterenkojul/Desktop/dh_project/images/117500500/117500500_13.png
/Users/nesterenkojul/Desktop/dh_project/images/